<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
li table, li div.course-figure { margin-left: 0 !important; }
.course-figure { width: auto; }
.katex-display { text-align: left !important; margin: 1rem 0 1.25rem 2.5rem !important; }
.katex-display > .katex, .katex-display > .katex > .katex-html { text-align: left !important; }
</style>

## 경제 예제: 사업 선택

### 1 · 문제와 정식화(problem formulation)

- 예산(budget) 7백만 원 안에서 사업을 골라 순편익(net benefit)을 최대화(maximization)한다.
- 사업별 효과는 서로 더할 수 있다고 가정한다.
- 아래 값은 가상 자료이며 단위는 백만 원이다.

| 사업 | 비용 (Cost) | 순편익 (Net benefit) |
|:---|:---|:---|
| A | 4 | 7 |
| B | 3 | 6 |
| C | 2 | 4 |
| D | 5 | 8 |

- 의사결정벡터(decision vector)는 $\boldsymbol{x}=(x_1,x_2,x_3,x_4)$이며, 각 성분(component)은 A·B·C·D의 실행 여부이다.
- $x_i=1$이면 사업을 실행하고, $x_i=0$이면 실행하지 않는다.
- 목적함수(objective function)는 선택한 사업의 순편익 합이다.
- 순편익에는 비용이 이미 반영되어 있으므로 비용을 다시 빼지 않는다.

$$
\max_{\boldsymbol{x}}\;7x_1+6x_2+4x_3+8x_4
$$

- 총비용(total cost)은 예산을 넘을 수 없다.

$$
4x_1+3x_2+2x_3+5x_4\leq7,\qquad
x_i\in\{0,1\}\;(i=1,2,3,4)
$$

- 그림에서 A·C는 비용 6으로 허용되고, A·D는 비용 9로 거부된다.

<div class="course-figure" style="text-align: left; margin: 0.65rem 0 1.5rem 2.5rem;">
<img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/3c17b9cda768a9d4f9aaca66bc0b7d01e882a5da/05-2_metaheuristic_optimization_genetic_algorithm/assets/budget_feasibility.svg" alt="Costs 6 and 9 are compared with the same budget limit of 7 million KRW." width="440" style="display: block; width: 27.5em; max-width: 100%; height: auto; margin: 0;">
</div>


<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
li table, li div.course-figure { margin-left: 0 !important; }
.course-figure { width: auto; }
.katex-display { text-align: left !important; margin: 1rem 0 1.25rem 2.5rem !important; }
.katex-display > .katex, .katex-display > .katex > .katex-html { text-align: left !important; }
</style>

### 2 · 코드 실행

- 공통 알고리듬(algorithm)을 정의한 뒤, 아래 셀에서 데이터(data)와 평가 함수(evaluation function)를 넣어 실행한다.


In [ ]:
import random

def genetic_algorithm(initial_population, gene_values, score, generations=20, seed=0):
    selection_top_k = 2
    crossover_rate = 1.0
    mutation_rate = 0.3
    elite_count = 1
    blend_weight = 0.5
    mutation_scale = 0.1
    bounds = (-2.0, 2.0)

    # gene_values: allowed discrete values; None: real-valued genes.
    rng = random.Random(seed)
    population = initial_population
    population_size = len(population)

    for generation in range(generations):
        # Rank all candidates from highest to lowest score.
        ranked = []
        remaining = population.copy()  # Removing entries must not change population.
        while len(remaining) > 0:
            best = remaining[0]
            for candidate in remaining:
                if score(candidate) > score(best):
                    best = candidate
            ranked.append(best)
            remaining.remove(best)

        # Elitism: keep the best candidates unchanged.
        next_population = []
        for i in range(elite_count):
            next_population.append(ranked[i])

        # Selection: choose each parent randomly from the top candidates.
        parents = ranked[:selection_top_k]
        while len(next_population) < population_size:
            parent_a = rng.choice(parents)
            parent_b = rng.choice(parents)
            child = parent_a.copy()  # Change the child, not the parent.

            # Crossover: weighted average for real genes; join parts otherwise.
            if rng.random() < crossover_rate:
                if gene_values is None:
                    for i in range(len(child)):
                        child[i] = (blend_weight * parent_a[i]
                                    + (1 - blend_weight) * parent_b[i])
                else:
                    cut = rng.randrange(1, len(child))
                    for i in range(cut, len(child)):
                        child[i] = parent_b[i]

            # Mutation: change one randomly selected gene.
            if rng.random() < mutation_rate:
                i = rng.randrange(len(child))
                if gene_values is None:
                    child[i] += rng.uniform(-mutation_scale, mutation_scale)
                    if child[i] < bounds[0]:
                        child[i] = bounds[0]
                    if child[i] > bounds[1]:
                        child[i] = bounds[1]
                else:
                    alternatives = []
                    for value in gene_values:
                        if value != child[i]:
                            alternatives.append(value)
                    child[i] = rng.choice(alternatives)

            # Additional constraints: replace an infeasible child with its parent.
            if score(child) == float("-inf"):
                child = parent_a
            next_population.append(child)

        population = next_population

    best = population[0]
    for candidate in population:
        if score(candidate) > score(best):
            best = candidate
    return best.copy()  # Keep the returned list independent of the input.


In [ ]:
costs = [4, 3, 2, 5]
benefits = [7, 6, 4, 8]
budget = 7

def evaluate(decision):
    cost = 0
    for i in range(4):
        cost += costs[i] * decision[i]
    if cost > budget:
        return {"cost": cost, "feasible": False, "net_benefit": None}
    benefit = 0
    for i in range(4):
        benefit += benefits[i] * decision[i]
    return {"cost": cost, "feasible": True, "net_benefit": benefit}

def score(decision):
    result = evaluate(decision)
    if result["feasible"]:
        return result["net_benefit"]
    return float("-inf")

initial_population = [[1, 0, 0, 0], [0, 1, 0, 0],
                      [0, 0, 1, 0], [0, 0, 0, 1]]
best = genetic_algorithm(initial_population, [0, 1], score, seed=0)
result = evaluate(best)
print("Decision:", best)
print("Cost (million KRW):", result["cost"])
print("Feasible:", result["feasible"])
print("Net benefit (million KRW):", result["net_benefit"])
print("Stop reason: generation limit")


<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
li table, li div.course-figure { margin-left: 0 !important; }
.course-figure { width: auto; }
.katex-display { text-align: left !important; margin: 1rem 0 1.25rem 2.5rem !important; }
.katex-display > .katex, .katex-display > .katex > .katex-html { text-align: left !important; }
</style>

- 결과는 C와 D를 선택한 조합(combination)이다.
- 총비용은 7백만 원이고 순편익은 12백만 원이다.
- 예산은 충족하지만, 순편익이 13백만 원인 A·B 조합보다 낮으므로 최적해(optimal solution)는 아니다.
